### American vs Euorpean Options
The key difference between American and European option lies in their boundary coditions.\
Black-Scholes European Option = PDE holds for: 0 $\leq$ $\tau$ $\leq$ T, S > 0 \
Black-Scholes American Option = PDE holds for: 0 $\leq$ $\tau$ $\leq$ T, 0 < S $\leq$ b($\nu$, $\tau$)

Where:\
T = Maturity\
$\tau$ = Time Remaining Until to Maturity\
$\nu$ = Volatility\
b($\nu$, $\tau$) = Critical threshold where if stok price crosses this boundary, the PDE no longer applies because it becomes mathematically optimal to exercise the option immediately rather than continue holding it.

### The Early Excercise Boundary
Barone-Adesi-Whaley(1987), recognizes an American option can be decomposed as:

$C_A = C_E + V$

European Call Price: $C_E$\
Early Exercise Premium: $V$


Numerical Methods to Solve for this Free Boundary:\
- Binomial Trees
- Finit Differences
- Monte Carlo with Regression

When would early excercise become desireable?
1. dividends that the owner of the stock will recieve
2. positive cashflow on interest earned

In the case that there are not dividend/interest considerations, then the value of the american option should equal the value of a European option.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import ipywidgets as widgets
# from IPython.display import display

# Black–Scholes European formulas
def bs_call(S, K, r, sigma, T):
    d1 = (np.log(S/K) + (r + 0.5 * sigma**2)*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2)

def bs_put(S, K, r, sigma, T):
    d1 = (np.log(S/K) + (r + 0.5 * sigma**2)*T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r*T) * norm.cdf(-d2) - S * norm.cdf(-d1)

# Binomial tree for American option
def american_binomial(S, K, r, sigma, T, N, opt_type="call"):
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1/u
    p = (np.exp(r * dt) - d) / (u - d)

    # stock prices
    ST = np.array([S * u**j * d**(N-j) for j in range(N+1)])

    # terminal payoff
    if opt_type == "call":
        V = np.maximum(ST - K, 0)
    else:
        V = np.maximum(K - ST, 0)

    # backward induction
    for i in range(N-1, -1, -1):
        ST = ST[1:] / u  # step down tree
        V = np.exp(-r*dt)*(p*V[1:] + (1-p)*V[:-1])
        intrinsic = (ST - K) if opt_type == "call" else (K - ST)
        V = np.maximum(V, intrinsic)

    return V[0]

# Parameters 
S_range = np.linspace(50, 150, 100)
K = 100
sigma = 0.25
T = 1.0
N = 200

# Create interactive plot function 
def plot_options(q, r):
    # Compute option values
    euro_calls = [bs_call(s, K, r - q, sigma, T) for s in S_range]
    euro_puts = [bs_put(s, K, r - q, sigma, T) for s in S_range]
    
    amer_calls = [american_binomial(s, K, r - q, sigma, T, N, "call") for s in S_range]
    amer_puts = [american_binomial(s, K, r - q, sigma, T, N, "put") for s in S_range]
    
    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    # CALLS 
    ax = axes[0]
    ax.plot(S_range, euro_calls, 'k--', label="European Call", linewidth=2)
    ax.plot(S_range, amer_calls, 'b-', label="American Call", linewidth=2)
    ax.set_title("Call Options", fontsize=14, fontweight='bold')
    ax.set_xlabel("Underlying Price ($)", fontsize=12)
    ax.set_ylabel("Option Value ($)", fontsize=12)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    # PUTS
    ax = axes[1]
    ax.plot(S_range, euro_puts, 'k--', label="European Put", linewidth=2)
    ax.plot(S_range, amer_puts, 'b-', label="American Put", linewidth=2)
    ax.set_title("Put Options", fontsize=14, fontweight='bold')
    ax.set_xlabel("Underlying Price ($)", fontsize=12)
    ax.set_ylabel("Option Value ($)", fontsize=12)
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'Interactive American vs European Option Pricing\nContinuous Dividend Yield: {q:.1%} | Interest Rate: {r:.1%}', 
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Create interactive widgets
q_slider = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=0.10,
    step=0.005,
    description='Dividend (q):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1%',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px')
)

r_slider = widgets.FloatSlider(
    value=0.02,
    min=-0.05,
    max=0.15,
    step=0.005,
    description='Interest Rate (r):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1%',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px')
)

widgets.interact(plot_options, q=q_slider, r=r_slider)

In theory, an American call on a non-dividend paying stock is equal to a European call. so the following conditions are when divergence becomes meaningful.
1. Intrinsic value is large (Deep ITM), and holding the call means missing out on dividend yield or holding onto the put meaning missing out on interest
2. High Dividend Yield (q), dividend stream is "carry cost" of holding stock via call.

Early exercise only matters when you gain more by taking the stock now than by keeping your optionality. Whether that's because of dividend considerations or possible interest earned.

### Scenarios
1. For American call options on non-dividend-paying stocks\
There is never a rational reason to exercise early.
2. For American call options on dividend-paying stocks\
Early exercise can be optimal right before an ex-dividend date.
3. For American put options\
Early exercise can many times be optimal.
 
Why Put Options:
- Deep-in-the-money puts behave like debt - by exercising early, you receive cash today rather than later.

- If interest rates are positive, receiving cash sooner can outweigh remaining time value.

- Deep-in-the-money + low volatility + longer maturity increases the chance that early exercise is optimal.

- There's no single "timestamp," but the early-exercise region is well-defined in models like the free-boundary solution of the American PDE.

## Synthetic Data Creation
- Synthetic data will allow us to create features that encompass every economically viable condition found in the market
- Each feature must have a different range, otherwise, the dataset will become over encoimpassing and too large for price generation.
- Since its impossible to combine every possible feature value with eachother sampling is important. We will create five matrices of values one for each feature and combine them to create the dataset.

#### Sampling
Without sampling when generating my dataset every possible increment of a value would need to be included. This is not possible, thus, we use sampling to generate thousands of these combinations automatically. The different methods listed determine whether these values are spread evenly, or are concentrated differently across values.

- **Uniform sampling:** Draws each value with equal probability acrosss the range of values. We use this for parameters who scale lineraly across the range and want none of the value to have more representation in the dataset over others.

- **Log-uniform sampling:** Draws each value with equal probability across log-space. We use this on features that change multiplicatly like volatility. Using something like **Uniform Sampling** would under represent low volatility values which are what is most typically found in regular market conditions.

#### Boundaries:
- **Moneyness (K/S): 0.50 to 1.50** covers deep ITM to deep OTM while excluding sub-0.50 territory where options behave like pure stock positions and add little training signal.

- **Time to Expiry (T): 0.005 to 3.0 years** uses 0.005 (~2 days) as the practical floor before binomial trees become numerically unstable, and 3.0 years captures LEAPS without wasting samples on the 3 to 5 year range where behavior barely changes.

- **Volatility (σ): 0.05 to 1.50** sets 0.05 to reflect the quietest liquid instruments and 1.50 to cover extreme events like biotech binary readouts or meme stocks, capturing the full spectrum of real-world vol regimes the model may encounter.

- **Risk-free rate (r): -0.02 to 0.10** sets -0.02 to reflect the negative rate environment seen in Europe and Japan post-2008, and 0.10 to cover aggressive tightening cycles like 2022 to 2023 without sampling rates that have never existed in modern developed markets.

- **Dividend yield (q): 0.00 to 0.06** spans from zero for growth stocks and non-payers up to 0.06 for the highest-yielding liquid equities with active options markets, ensuring adequate coverage of cases where q > r which triggers rational early exercise on American calls.




In [ ]:
N = 100_000
lo, hi =0.05, 1.50  # using volatility as the example

uniform_samples= np.random.uniform(lo, hi, N)
log_uniform_samples = np.exp(np.random.uniform(np.log(lo), np.log(hi), N))

fig, axes = plt.subplots(2, 1, figsize=(6, 8))

axes[0].hist(uniform_samples, bins=80, color="steelblue", edgecolor="none")
axes[0].set_title("Uniform Sampling (e.g. moneyness, rates)", fontsize=13)
axes[0].set_xlabel("Value")
axes[0].set_ylabel("Count")

axes[1].hist(log_uniform_samples, bins=80, color="purple", edgecolor="none")
axes[1].set_title("Log-Uniform Sampling (e.g. volatility, time)", fontsize=13)
axes[1].set_xlabel("Value")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [110]:
import pandas as pd

def generateOptionsGrid(n_random=50000):

    mon_range= (0.50, 1.50)
    time_range= (0.005, 3.0)
    vol_range= (0.05, 1.50)
    rfr_range= (-0.02, 0.10)
    div_range= (0.00, 0.06)

    np.random.seed(42)
    N = n_random

    S = 100.0
    moneyness = np.random.uniform(*mon_range, N)

    data = pd.DataFrame({
        'S':np.full(N, S),
        'K':S * moneyness,
        'T':np.exp(np.random.uniform(np.log(time_range[0]), np.log(time_range[1]), N)),
        'r':np.random.uniform(*rfr_range, N),
        'sigma':np.exp(np.random.uniform(np.log(vol_range[0]), np.log(vol_range[1]), N)),
        'q':np.random.uniform(*div_range, N),
        'option_type': np.random.choice([1, 0], N)
    })

    # Systematic grid to guarantee coverage of key regimes
    moneyness_cases = []
    for m in [0.5, 0.7, 0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.5]:
        for T in [1/52, 1/12, 0.25, 0.5, 1.0, 2.0]:
            for sigma in [0.1, 0.2, 0.4, 0.8]:
                for r in [0.01, 0.04, 0.07]:
                    moneyness_cases.append({
                        'S': S, 'K': S * m, 'T': T, 'r': r,
                        'sigma': sigma, 'q': 0.02,
                        'option_type': np.random.choice([1, 0])
                    })

    systematic = pd.DataFrame(moneyness_cases)
    df = pd.concat([data, systematic], ignore_index=True)

    return df.reset_index(drop=True)

In [ ]:
df = generateOptionsGrid(500000)
df.head(20)

In [ ]:


fig, axes = plt.subplots(2, 3, figsize=(12, 4))

axes[0][0].hist(df["S"], bins=80, color="steelblue", edgecolor="none")
axes[0][0].set_title("Spot Price", fontsize=13)
axes[0][0].set_xlabel("Value")
axes[0][0].set_ylabel("Count")

axes[0][1].hist(df["K"], bins=80, color="purple", edgecolor="none")
axes[0][1].set_title("Moneyness", fontsize=13)
axes[0][1].set_xlabel("Value")
axes[0][1].set_ylabel("Count")

axes[0][2].hist(df["T"], bins=80, color="steelblue", edgecolor="none")
axes[0][2].set_title("Days until Expiration", fontsize=13)
axes[0][2].set_xlabel("Value")
axes[0][2].set_ylabel("Count")

axes[1][0].hist(df["r"], bins=80, color="purple", edgecolor="none")
axes[1][0].set_title("Risk-Free Rate", fontsize=13)
axes[1][0].set_xlabel("Value")
axes[1][0].set_ylabel("Count")

axes[1][1].hist(df["sigma"], bins=80, color="steelblue", edgecolor="none")
axes[1][1].set_title("Volatility", fontsize=13)
axes[1][1].set_xlabel("Value")
axes[1][1].set_ylabel("Count")

axes[1][2].hist(df["q"], bins=80, color="purple", edgecolor="none")
axes[1][2].set_title("Dividend", fontsize=13)
axes[1][2].set_xlabel("Value")
axes[1][2].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

corr_matrix = df.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f") 
plt.title('Correlation Matrix Heatmap')
plt.show()

This signals that none of the features are correlated, which we were hoping to accompkish with this data.

## Lets Compare the Synthetic Data to SPY Options Chain

In [ ]:
data = pd.read_csv('archive.zip')
data.head()


In [ ]:
pd.set_option('display.max_columns', None)
data.head()

In [ ]:
data.describe()

In [ ]:
print(data.columns.tolist())

In [ ]:
data.columns = data.columns.str.strip().str.replace('[', '', regex=False).str.replace(']', '', regex=False)

data["DTE"] = pd.to_numeric(data["DTE"], errors="coerce")
data["STRIKE"] = pd.to_numeric(data["STRIKE"], errors="coerce")
data["UNDERLYING_LAST"] = pd.to_numeric(data["UNDERLYING_LAST"], errors="coerce")
data["C_IV"] = pd.to_numeric(data["C_IV"], errors="coerce")

data["DTE_YEARS"] = data["DTE"] / 365
data["MONEYNESS"] = data["STRIKE"] / data["UNDERLYING_LAST"]

data = data[
    (data["UNDERLYING_LAST"] > 0) &
    (data["STRIKE"] > 0) &
    (data["DTE"] > 0) &
    (data["DTE"] < 3650) &
    (data["C_IV"] > 0) &
    (data["C_IV"] < 3) &
    (data["MONEYNESS"] > 0) &
    (data["MONEYNESS"] < 3)
]

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0][0].hist(data["UNDERLYING_LAST"].dropna(), bins=80, color="steelblue", edgecolor="none")
axes[0][0].set_title("Spot Price", fontsize=13)
axes[0][0].set_xlabel("Value")
axes[0][0].set_ylabel("Count")

axes[0][1].hist(data["MONEYNESS"].dropna(), bins=80, color="purple", edgecolor="none")
axes[0][1].set_title("Moneyness", fontsize=13)
axes[0][1].set_xlabel("Value")
axes[0][1].set_ylabel("Count")

axes[1][0].hist(data["C_IV"].dropna(), bins=80, color="steelblue", edgecolor="none")
axes[1][0].set_title("Call Implied Volatility", fontsize=13)
axes[1][0].set_xlabel("Value")
axes[1][0].set_ylabel("Count")

axes[1][1].hist(data["DTE_YEARS"].dropna(), bins=80, color="steelblue", edgecolor="none")
axes[1][1].set_title("Time to Expiration (Years)", fontsize=13)
axes[1][1].set_xlabel("Value")
axes[1][1].set_ylabel("Count")



plt.tight_layout()
plt.show()

## Spot Price Distribution
This histogram shows the distribution of underlying asset prices in the real option chain dataset. The data spans a wide range of prices, reflecting the many different market conditions under which options are traded. Understanding the range of underlying prices helps ensure that the synthetic dataset used for model training reflects realistic market scenarios.

## Moneyness Distribution 
This plot shows the distribution of option moneyness, defined as strike price divided by the underlying asset price. The distribution peaks near 1.0, indicating that most traded options are close to at the money, while fewer options exist far in or out of the money.

## Call Implied Volatilty Distribution 
This histogram shows the distribution of implied volatility for call options. Most observations fall in the lower volatility range, with a long tail of higher volatility values representing periods of increased uncertainty or extreme market conditions.

## Time to Expiration 
This histogram shows the distribution of time to expiration in years. Most options have relatively short maturities, while fewer contracts extend to longer expiration horizons.

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(data["MONEYNESS"], data["C_IV"], s=1, alpha=0.2)
plt.xlabel("Moneyness")
plt.ylabel("Call Implied Volatility")
plt.title("Volatility Smile")
plt.show()

## Volatility Smile
This scatter plot shows the relationship between moneyness and implied volatility. The curved shape illustrates the well-known volatility smile, where implied volatility tends to be higher for deep in-the-money and out-of-the-money options compared to at-the-money options

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(data["DTE_YEARS"], data["C_IV"], s=1, alpha=0.2)
plt.xlabel("Time to Expiration (Years)")
plt.ylabel("Call Implied Volatility")
plt.title("Volatility vs Time to Expiration")
plt.show()

## Volatility vs Time to Expiration
This plot shows how implied volatility varies across different times to expiration. Shorter maturities tend to exhibit greater volatility dispersion, while longer maturities show more stable volatility levels.

In [ ]:
sample = data.sample(200000, random_state=42).copy()

moneyness_bins = np.linspace(0.7, 1.3, 30)
time_bins = np.linspace(0.01, 2.0, 30)

sample["m_bin"] = pd.cut(sample["MONEYNESS"], moneyness_bins)
sample["t_bin"] = pd.cut(sample["DTE_YEARS"], time_bins)

surface = sample.groupby(["m_bin","t_bin"], observed=False)["C_IV"].mean().unstack()

plt.figure(figsize=(9,6))

plt.imshow(
    surface.values,
    aspect="auto",
    origin="lower",
    cmap="viridis",
    extent=[time_bins[0], time_bins[-1], moneyness_bins[0], moneyness_bins[-1]]
)

plt.colorbar(label="Implied Volatility")

plt.xlabel("Time to Expiration (Years)")
plt.ylabel("Moneyness")
plt.title("Implied Volatility Surface")

plt.show()

## Implied Volatility Surface Heatmap
This heatmap visualizes the implied volatility surface across both moneyness and time to expiration. The color intensity represents average implied volatility at each region of the surface, revealing how volatility changes across strike levels and maturities

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

sample = data.sample(200000, random_state=42).copy()

moneyness_bins = np.linspace(0.6, 1.4, 30)
time_bins = np.linspace(0.01, 3, 30)

sample["m_bin"] = pd.cut(sample["MONEYNESS"], moneyness_bins)
sample["t_bin"] = pd.cut(sample["DTE_YEARS"], time_bins)

surface = sample.groupby(["m_bin", "t_bin"], observed=False)["C_IV"].mean().unstack()

X = np.array([b.mid for b in surface.columns])
Y = np.array([b.mid for b in surface.index])
X, Y = np.meshgrid(X, Y)
Z = surface.values

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

surf = ax.plot_surface(X, Y, Z, cmap="viridis", linewidth=0, antialiased=True)

ax.set_xlabel("Time to Expiration (Years)", labelpad=12)
ax.set_ylabel("Moneyness", labelpad=12)
ax.set_zlabel("Implied Volatility", labelpad=18)

ax.set_title("Implied Volatility Surface", pad=20)

ax.view_init(elev=25,azim=-60)

fig.colorbar(surf, shrink=0.6, aspect=12, pad=0.12)

plt.subplots_adjust(left=0.02,right=0.88,top=0.90, bottom=0.05)
plt.show()

## Revision of Synthetic Price Generation

### New synthetic data set
- improve spot range to random sample values from 80 - 120
- improve call/put logic it now creates a dataset for calls and one for puts then merges them together

In [62]:
def generateOptionsGrid(n_random=50000):
    spot_range = (80,120)
    mon_range= (0.50, 1.50)
    time_range= (0.005, 3.0)
    vol_range= (0.05, 1.50)
    rfr_range= (-0.02, 0.10)
    div_range= (0.00, 0.06)

    np.random.seed(42)
    N = n_random

    S = 100.0
    moneyness = np.random.uniform(*mon_range, N)

    data = pd.DataFrame({
        'S':np.random.uniform(spot_range[0], spot_range[1], N),
        'K':S * moneyness,
        'T':np.exp(np.random.uniform(np.log(time_range[0]), np.log(time_range[1]), N)),
        'r':np.random.uniform(*rfr_range, N),
        'sigma':np.exp(np.random.uniform(np.log(vol_range[0]), np.log(vol_range[1]), N)),
        'q':np.random.uniform(*div_range, N),
    })
    
    calls = data.assign(option_type=1)
    puts  = data.assign(option_type=0)
    data  = pd.concat([calls, puts], ignore_index=True)
    # Systematic grid to guarantee coverage of key regimes
    moneyness_cases = []
    for m in [0.5, 0.7, 0.8, 0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 1.5]:
        for T in [1/52, 1/12, 0.25, 0.5, 1.0, 2.0]:
            for sigma in [0.1, 0.2, 0.4, 0.8]:
                for r in [0.01, 0.04, 0.07]:
                    moneyness_cases.append({
                        'S': S, 'K': S * m, 'T': T, 'r': r,
                        'sigma': sigma, 'q': 0.02,
                        'option_type': np.random.choice([1, 0])
                    })

    systematic = pd.DataFrame(moneyness_cases)
    df = pd.concat([data, systematic], ignore_index=True)

    return df.reset_index(drop=True)

In [ ]:
df = generateOptionsGrid()
df.head(5)


In [ ]:
df.describe()

In [ ]:
import matplotlib.colors as mcolors

df = generateOptionsGrid()

df['moneyness'] = df['K'] / df['S']
monbins= [0.50, 0.75, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.20, 1.50]
timebins =[0.00, 1/52, 1/12, 0.25, 0.5, 1.0, 2.0, 3.0]

df['mon_bin']= pd.cut(df['moneyness'], bins=monbins)
df['time_bin'] = pd.cut(df['T'],bins=timebins)

pivot = df.pivot_table(index='time_bin', columns='mon_bin', values='S', aggfunc='count')

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', norm=mcolors.LogNorm())

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([str(c) for c in pivot.columns], rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([str(i) for i in pivot.index], fontsize=9)

for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{int(val):,}', ha='center', va='center',
                    fontsize=8, color='black' if val < 5000 else 'white')

plt.colorbar(im, ax=ax, label='Record count')
ax.set_xlabel('Moneyness bucket')
ax.set_ylabel('Time to expiry bucket')
ax.set_title('moneyness × time to expiry')
plt.tight_layout()
plt.show()

### Method to generate random sample options grid

In [91]:
import numpy as np
import pandas as pd
import QuantLib as ql
from math import ceil

import numpy as np
import pandas as pd

def generateOptionsGrid(n_random=50000):
    spot_range = (80, 120)
    time_range = (0.005, 3.0)
    vol_range = (0.05, 1.50)
    rfr_range = (-0.02, 0.10)
    div_range = (0.00, 0.06)

    np.random.seed(42)

    S_val = 100.0

    # 55% broadly sampled,  25% near expiriation, 20% near at the money 
    broad = int(n_random * 0.55)
    near_exp = int(n_random * 0.25)
    near_ATM = n_random - broad - near_exp

    # Broad Random Sampling
    S_broad = np.random.uniform(spot_range[0], spot_range[1], broad)
    m_broad = np.exp(np.random.uniform(np.log(0.50), np.log(1.50), broad))
    data_broad = pd.DataFrame({
        'S': S_broad,
        'K': S_broad * m_broad,
        'T': np.exp(np.random.uniform(np.log(time_range[0]), np.log(time_range[1]), broad)),
        'r': np.random.uniform(*rfr_range, broad),
        'sigma': np.exp(np.random.uniform(np.log(vol_range[0]), np.log(vol_range[1]), broad)),
        'q': np.random.uniform(*div_range, broad),

    })

    #near expiration options across the full moneyness range
    S_exp = np.random.uniform(85, 115, near_exp)
    m_exp = np.exp(np.random.uniform(np.log(0.80), np.log(1.20), near_exp))
    data_hard = pd.DataFrame({
        'S': S_exp,
        'K': m_exp * m_exp,
        'T': np.exp(np.random.uniform(np.log(0.003), np.log(0.10), near_exp)),
        'r': np.random.uniform(*rfr_range, near_exp),
        'sigma': np.exp(np.random.uniform(np.log(0.05), np.log(0.60), near_exp)),
        'q': np.random.uniform(*div_range, near_exp),
    })

    # Near at the money block
    S_near = np.random.uniform(spot_range[0], spot_range[1], near_ATM)
    m_near = np.exp(np.random.normal(0, 0.05, near_ATM))
    m_near = np.clip(m_near, 0.70, 1.30)
    data_atm = pd.DataFrame({
        'S': S_near,
        'K': S_near * m_near,
        'T': np.exp(np.random.uniform(np.log(time_range[0]), np.log(time_range[1]), near_ATM)),
        'r': np.random.uniform(*rfr_range, near_ATM),
        'sigma': np.exp(np.random.uniform(np.log(0.05), np.log(0.80), near_ATM)),
        'q': np.random.uniform(*div_range, near_ATM),
    })

    data = pd.concat([data_broad, data_hard, data_atm], ignore_index=True)

    # K realistic range
    data['K'] = data['K'].clip(lower=5.0)
    data['log_moneyness'] = np.log(data['S'] / data['K'])
    data['sqrt_T'] = np.sqrt(data['T'])

    calls = data.assign(option_type=1)
    puts= data.assign(option_type=0)
    data= pd.concat([calls, puts], ignore_index=True)

    # both calls and puts for each parameter combo
    moneyness_cases = []
    for m in [0.5, 0.7, 0.8, 0.85, 0.90, 0.95, 0.98, 1.0, 1.02, 1.05, 1.10, 1.15, 1.2, 1.5]:
        for T in [1/252, 3/252, 1/52, 1/12, 0.25, 0.5, 1.0, 2.0]:
            for sigma in [0.05, 0.10, 0.20, 0.35, 0.50, 0.80, 1.20]:
                for r in [0.00, 0.02, 0.05, 0.08]:
                    for opt_type in [1, 0]:
                        moneyness_cases.append({
                            'S': S_val, 'K': S_val * m, 'T': T, 'r': r,
                            'sigma': sigma, 'q': 0.02,
                            'option_type': opt_type,
                            'log_moneyness': np.log(1.0 / m),
                            'sqrt_T': np.sqrt(T),
                        })

    systematic = pd.DataFrame(moneyness_cases)
    df = pd.concat([data, systematic], ignore_index=True)
    
    df['moneyness'] = df['S'] / df['K']

    df['intrinsic_value_raw'] = np.where(
        df['option_type'] == 1, 
        np.maximum(df['S'] - df['K'], 0),  # Call
        np.maximum(df['K'] - df['S'], 0)   # Put
    )
    df['intrinsic_normalized'] = df['intrinsic_value_raw'] / df['K']

    df['total_variance'] = (df['sigma'] ** 2) * df['T']
    df['vol_sqrt_T'] = df['sigma'] * np.sqrt(df['T'])

    # Cost of Carry
    df['cost_of_carry'] = df['r'] - df['q']
    return df.reset_index(drop=True)

In [ ]:
df = generateOptionsGrid()
df.head(5)

### Method to add price column

In [93]:
import QuantLib as ql
def price_american_crr(S, K, T, r, sigma, q, option_type, steps=200):
    today = ql.Date.todaysDate()
    ql.Settings.instance().evaluationDate = today

    maturity = today + max(1, int(ceil(T * 365)))

    payoff_type = ql.Option.Call if int(option_type) == 1 else ql.Option.Put

    exercise = ql.AmericanExercise(today, maturity)
    payoff = ql.PlainVanillaPayoff(payoff_type, float(K))
    option = ql.VanillaOption(payoff, exercise)

    spot = ql.QuoteHandle(ql.SimpleQuote(float(S)))
    dividend_ts = ql.YieldTermStructureHandle(
        ql.FlatForward(today, float(q), ql.Actual365Fixed())
    )
    riskfree_ts = ql.YieldTermStructureHandle(
        ql.FlatForward(today, float(r), ql.Actual365Fixed())
    )
    vol_ts = ql.BlackVolTermStructureHandle(
        ql.BlackConstantVol(today, ql.NullCalendar(), float(sigma), ql.Actual365Fixed())
    )

    process = ql.BlackScholesMertonProcess(spot, dividend_ts, riskfree_ts, vol_ts)
    engine = ql.FdBlackScholesVanillaEngine(process, 200, 400)  #Switched to finite difference method industry standard when also considering hedging delta and gamma
    option.setPricingEngine(engine)

    return option.NPV()




### Method that calculates black scholes columns 

In [94]:
import numpy as np
import pandas as pd
from scipy.stats import norm

def add_black_scholes_features(df):
    S, K, T, r, sigma, q = df['S'], df['K'], df['T'], df['r'], df['sigma'], df['q']
    
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    nd1 = norm.cdf(d1)
    nd2 = norm.cdf(d2)
    N_minus_d1 = norm.cdf(-d1)
    N_minus_d2 = norm.cdf(-d2)
    
    call_price = S * np.exp(-q * T) * nd1 - K * np.exp(-r * T) * Nd2
    put_price = K * np.exp(-r * T) * N_minus_d2 - S * np.exp(-q * T) * N_minus_d1
    
    df['bs_price'] = np.where(df['option_type'] == 1, call_price, put_price)
    
    # Adding Delta for extra structural hints to the DNN
    call_delta = np.exp(-q * T) * nd2
    put_delta = -np.exp(-q * T) * N_minus_d1
    df['bs_delta'] = np.where(df['option_type'] == 1, call_delta, put_delta)
    
    return df



### Calculate Early excercise premium from American and European prices and add higher order greeks

In [95]:
def prepare_dnn_dataset(df):
    df = df.copy()
    # Early Exercise Premium 
    df['EEP'] = df['american_price'] - df['bs_price']
    
    # Scale everything by Strike price
    df['norm_price'] = df['american_price'] / df['K']
    df['norm_bs_price'] = df['bs_price'] / df['K']
    df['norm_EEP'] = df['EEP'] / df['K']  # Target
    
    
    df['moneyness'] = df['S'] / df['K']
    df['time_scaled_variance'] = (df['sigma'] ** 2) * df['T']
    df['cost_of_carry'] = df['r'] - df['q']
    
    # Calculate Intrinsic Value
    intrin= np.where(
        df['option_type'] == 1, 
        np.maximum(df['S'] - df['K'], 0), 
        np.maximum(df['K'] - df['S'], 0)
    )
    df['normalized_intrinsic'] = intrin / df['K']

    # Advanced Greeks Calculations
    d1 = (np.log(df['S'] / df['K']) + (df['r'] - df['q'] + 0.5 * df['sigma']**2) * df['T']) / (df['sigma'] * np.sqrt(df['T']))
    d2 = d1 - df['sigma'] * np.sqrt(df['T'])
    
    df['bs_gamma'] = (norm.pdf(d1) * np.exp(-df['q'] *df['T'])) / (df['S'] * df['sigma'] * np.sqrt(df['T']))
    df['bs_vega'] = (df['S'] * np.exp(-df['q'] *df['T']) * norm.pdf(d1) * np.sqrt(df['T'])) / 100.0
    theta_call = (- (df['S'] * df['sigma'] * np.exp(-df['q'] * df['T']) * norm.pdf(d1)) / (2 * np.sqrt(df['T'])) 
                  - df['r'] * df['K'] * np.exp(-df['r'] * df['T']) * norm.cdf(d2) 
                  + df['q'] * df['S'] * np.exp(-df['q'] * df['T']) * norm.cdf(d1)) / 365.0
                  
    theta_put = (- (df['S'] * df['sigma'] * np.exp(-df['q'] * df['T']) * norm.pdf(d1)) / (2 * np.sqrt(df['T'])) 
                 + df['r'] * df['K'] * np.exp(-df['r'] * df['T']) * norm.cdf(-d2) 
                 - df['q'] * df['S'] * np.exp(-df['q'] * df['T']) * norm.cdf(-d1)) / 365.0
    
    df['bs_theta'] = np.where(df['option_type'] == 1, theta_call, theta_put)

    df['strike_distance_pct'] = (df['K'] - df['S']) / df['S']
    return df
    
    print("Dataset preparation complete.")
    return df

In [ ]:
df.head()

### DNN Initialization

In [97]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np

class surrogateDNN(nn.Module):
    def __init__(self, input_size, output_size):
        super(surrogateDNN, self).__init__() 
        self.net = nn.Sequential(
            nn.Linear(input_size, 400),
            nn.SiLU(), 
            nn.Linear(400, 400),
            nn.SiLU(),
            nn.Linear(400, 400),
            nn.SiLU(),
            nn.Linear(400, 400),
            nn.SiLU(),
            nn.Linear(400, 400),
            nn.SiLU(),
            nn.Linear(400, 400),
            nn.SiLU(),
            nn.Linear(400, output_size)
        )

    def forward(self, x):
        return self.net(x)

# Method for preparing data for training
def prepare_data(df, target_col, batch_size=256): 
    # drop intermediate price columns to prevent data leakage
    cols_to_drop = [target_col, 'american_price', 'bs_price', 'EEP', 'norm_price']
    X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    y = df[target_col].values.reshape(-1, 1) 

    indices = np.arange(len(df))
    
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X, y, indices, test_size=0.2, random_state=42
    )

    # the raw Strike (K) and BS Price specifically for the test set evaluation later and for descaling    
    K_test = df['K'].values[idx_test].reshape(-1, 1)
    bs_price_test = df['bs_price'].values[idx_test].reshape(-1, 1)
    option_type_test = df['option_type'].values[idx_test].reshape(-1, 1) #extract option type for mertons rule
    q_test = df['q'].values[idx_test].reshape(-1, 1) #extract dividend for mertons rule

    x_scaler = StandardScaler()
    X_train_scaled = x_scaler.fit_transform(X_train)
    X_test_scaled = x_scaler.transform(X_test) 

    # Scale Targets
    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train)
    y_test_scaled = y_scaler.transform(y_test)

    # Convert to Tensors
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32) 
    y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

    # Create Loaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


    return train_loader, test_loader, X.shape[1], y_scaler, x_scaler, K_test, bs_price_test, option_type_test, q_test


        

In [ ]:
a=pd.read_csv("spy.csv", nrows= 100)
a.head()

## Now that we built a mathematically exceptional model we need to tweak it to perform well in actual market conditions
#### Higher Order Greeks
- Gamma tells the network how fast Delta is changing (the curvature of the price surface).

- Theta tells the network how heavily the option is decaying (critical for the Early Exercise Premium, as time value is the barrier to early exercise).

- Vega tells the network how sensitive the premium is to the Volatility input.

#### Volatility Surface
- While your network sees the IV, feeding it the explicit [STRIKE_DISTANCE_PCT] gives it spatial awareness of where this option sits on the volatility smile.

In [ ]:
print("Generating parameter")
raw_grid_df = generateOptionsGrid(n_random=50000)
price_df = add_black_scholes_features(raw_grid_df)
print("Pricing")
price_df["american_price"] = df.apply(
        lambda x: price_american_crr(
            x["S"], x["K"], x["T"], x["r"], x["sigma"], x["q"], x["option_type"], 200),axis=1)
print("Prepare dataset with computed columns")
df = prepare_dnn_dataset(price_df) #Root Mean Squared Error (RMSE): $0.0247
df.head()

## Run Model

In [ ]:
if __name__ == "__main__":

    target_column_name = 'normalized_EEP'
    
    train_loader, test_loader, input_features, y_scaler, x_scaler, K_test, bs_price_test, option_type_test, q_test = prepare_data(df, target_col=target_column_name)
    
    model = surrogateDNN(input_size=input_features, output_size=1)
    criterion = nn.HuberLoss()
    
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

    EPOCHS = 100
    print(f"Epoch {EPOCHS} ")
    
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        avg_train_loss = epoch_loss / len(train_loader)
        scheduler.step(avg_train_loss) 
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{EPOCHS}], Huber Loss: {avg_train_loss:.6f}')

    model.eval()

    all_preds_list = []
    all_targets_list = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            all_preds_list.append(outputs)
            all_targets_list.append(batch_y)

    # Concatenate into a single numpy array
    preds_combined = torch.cat(all_preds_list).numpy()
    targets_combined = torch.cat(all_targets_list).numpy()

    # Unscale
    unscaled_preds_normalized_EEP = y_scaler.inverse_transform(preds_combined)
    unscaled_targets_normalized_EEP = y_scaler.inverse_transform(targets_combined)

    unscaled_preds_normalized_EEP = np.maximum(unscaled_preds_normalized_EEP, 0)

    # Turn back into raw dollars
    predicted_EEP_dollars = unscaled_preds_normalized_EEP * K_test
    true_EEP_dollars = unscaled_targets_normalized_EEP * K_test

    # Reconstruct the American Option Price 
    predicted_american_price = bs_price_test + predicted_EEP_dollars
    true_american_price = bs_price_test + true_EEP_dollars

    # predicted price to equal the exact Black-Scholes price
    merton_mask = (option_type_test == 1) & np.isclose(q_test, 0.0)
    predicted_american_price[merton_mask] = bs_price_test[merton_mask]

    #Metrics
    mae = np.mean(np.abs(predicted_american_price - true_american_price))
    mse = np.mean((predicted_american_price - true_american_price)**2)
    rmse = np.sqrt(mse)
    max_sq_error = np.max((predicted_american_price - true_american_price)**2)
    within_penny = np.mean(np.abs(predicted_american_price - true_american_price) <= 0.05) * 100

    print(f'Mean Absolute Error: ${mae:.4f}')
    print(f'Root Mean Squared Error: ${rmse:.4f}')
    print(f'Max Squared Error: ${max_sq_error:.4f}')
    print(f'Within $0.05 absolute error: {within_penny:.1f}%')
    

In [102]:
spy = pd.read_csv("spy.csv", nrows=1000)


In [ ]:
spy.head()

In [ ]:
print(df.columns.tolist())

In [ ]:
print(spy.columns.tolist())

In [ ]:
print(df["sigma"])

### Clean Spy data so columns match sythetic data

In [ ]:

# Clean SPY headers
spy.columns = spy.columns.str.replace(r'[\[\]]', '', regex=True).str.strip()

#Extract Calls and Puts, rename to base variables, and calculate Midpoint
calls = spy[['QUOTE_DATE', 'UNDERLYING_LAST', 'STRIKE', 'DTE','C_IV', 'C_BID', 'C_ASK']].copy()
calls = calls.rename(columns={'QUOTE_DATE': 'date', 'UNDERLYING_LAST': 'S', 'STRIKE': 'K', 'C_IV': 'sigma'})
calls['option_type'] = 1
calls['market_price'] = (calls['C_BID'] + calls['C_ASK']) / 2.0

puts = spy[['QUOTE_DATE', 'UNDERLYING_LAST', 'STRIKE', 'DTE', 'P_IV', 'P_BID', 'P_ASK']].copy()

puts = puts.rename(columns={'QUOTE_DATE': 'date', 'UNDERLYING_LAST': 'S', 'STRIKE': 'K', 'P_IV': 'sigma'})
puts['option_type'] = 0
puts['market_price'] = (puts['P_BID'] + puts['P_ASK']) / 2.0


f = pd.concat([calls, puts], ignore_index=True)

# Force columns to be numeric
df['sigma'] = pd.to_numeric(df['sigma'], errors='coerce')
df['T'] = pd.to_numeric(df['T'], errors='coerce')
df = df[(df['T'] > 0) & (df['sigma'] > 0)].copy()

#Dynamic Interest Rate and Dividend based on the Date
df['date'] = pd.to_datetime(df['date'])
year = df['date'].dt.year
month = df['date'].dt.month

df['r'] = 0.020
df.loc[year == 2019, 'r'] = 0.022
df.loc[(year == 2020) & (month < 3), 'r'] = 0.015
df.loc[year.isin([2020, 2021]) & ~((year == 2020) & (month < 3)), 'r'] = 0.001
df.loc[(year == 2022) & (month <= 3), 'r'] = 0.005
df.loc[(year == 2022) & (month > 3) & (month <= 6), 'r'] = 0.015
df.loc[(year == 2022) & (month > 6) & (month <= 9), 'r'] = 0.030
df.loc[(year == 2022) & (month > 9), 'r'] = 0.040

df['q'] = 0.014
df.loc[year == 2019,'q'] = 0.0185
df.loc[year == 2020,'q'] = 0.0175
df.loc[year == 2021,'q'] = 0.0130
df.loc[year == 2022,'q'] = 0.0150

# Add cols to match the Synthetic Set
df['moneyness'] = df['S'] / df['K']
df['log_moneyness'] = np.log(df['moneyness'])
df['sqrt_T'] = np.sqrt(df['T'])
df['total_variance'] = (df['sigma'] ** 2) * df['T']
df['vol_sqrt_T'] = df['sigma'] * df['sqrt_T']
df['cost_of_carry'] = df['r'] - df['q']

# Intrinsic Value 
df['intrinsic_value_raw'] = np.where(
    df['option_type'] == 1,
    np.maximum(df['S'] - df['K'], 0),
    np.maximum(df['K'] - df['S'], 0)
)
df['intrinsic_normalized'] = df['intrinsic_value_raw'] / df['K']

target_cols = [
    'S', 'K', 'T', 'r', 'sigma', 'q', 'log_moneyness', 'sqrt_T', 
    'option_type', 'moneyness', 'intrinsic_value_raw', 
    'intrinsic_normalized', 'total_variance', 'vol_sqrt_T', 'cost_of_carry'
]

clean_spy_df = df[[ 'market_price'] + target_cols].dropna()
